In [3]:
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report)

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

os.makedirs('../results', exist_ok=True)

LABEL_MAP = {0: 'Benign', 1: 'DoS', 2: 'DDoS', 3: 'Mirai', 4: 'Spoofing'}

X_train: (1062114, 30)
X_test: (265529, 30)


In [4]:
from sklearn.preprocessing import LabelEncoder

le_binary = LabelEncoder()
y_train_binary_enc = le_binary.fit_transform(y_train_binary)  # Attack=0, Benign=1 (alphabetical) or check le_binary.classes_
y_test_binary_enc = le_binary.transform(y_test_binary)

print(le_binary.classes_)  # confirms which index maps to which label

stage1_models = {
    'DT': DecisionTreeClassifier(random_state=42),
    'RF': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGB': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    'LGBM': LGBMClassifier(random_state=42, n_jobs=-1)
}

stage1_results = {}
stage1_preds = {}

for name, model in stage1_models.items():
    print(f"\n--- Stage 1: {name} ---")
    t0 = time.time()
    model.fit(X_train, y_train_binary_enc)
    train_time = time.time() - t0

    y_pred_enc = model.predict(X_test)
    y_pred = le_binary.inverse_transform(y_pred_enc)  # back to 'Benign'/'Attack' strings
    stage1_preds[name] = y_pred

    acc = accuracy_score(y_test_binary, y_pred)
    prec = precision_score(y_test_binary, y_pred, pos_label='Attack')
    rec = recall_score(y_test_binary, y_pred, pos_label='Attack')
    f1 = f1_score(y_test_binary, y_pred, pos_label='Attack')

    cm = confusion_matrix(y_test_binary, y_pred, labels=['Benign', 'Attack'])
    fn = cm[1][0]
    total_attacks = cm[1][0] + cm[1][1]
    fnr = fn / total_attacks if total_attacks > 0 else 0

    stage1_results[name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'confusion_matrix': cm.tolist(), 'false_negative_rate': fnr,
        'train_time_sec': train_time
    }

    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print(f"Confusion Matrix [Benign, Attack]:\n{cm}")
    print(f"False Negative Rate (attacks missed): {fnr:.4f}")
    print(classification_report(y_test_binary, y_pred))

['Attack' 'Benign']

--- Stage 1: DT ---
Accuracy: 0.8926 | Precision: 0.9220 | Recall: 0.9249 | F1: 0.9234
Confusion Matrix [Benign, Attack]:
[[ 65028  14546]
 [ 13968 171987]]
False Negative Rate (attacks missed): 0.0751
              precision    recall  f1-score   support

      Attack       0.92      0.92      0.92    185955
      Benign       0.82      0.82      0.82     79574

    accuracy                           0.89    265529
   macro avg       0.87      0.87      0.87    265529
weighted avg       0.89      0.89      0.89    265529


--- Stage 1: RF ---
Accuracy: 0.9047 | Precision: 0.9376 | Recall: 0.9256 | F1: 0.9315
Confusion Matrix [Benign, Attack]:
[[ 68117  11457]
 [ 13843 172112]]
False Negative Rate (attacks missed): 0.0744
              precision    recall  f1-score   support

      Attack       0.94      0.93      0.93    185955
      Benign       0.83      0.86      0.84     79574

    accuracy                           0.90    265529
   macro avg       0.88      

In [6]:
#stage 2: attack type classfier 
le_attack = LabelEncoder()
y_train_attack_enc = le_attack.fit_transform(y_train_attack)  # 1,2,3,4 -> 0,1,2,3
y_test_attack_enc = le_attack.transform(y_test_attack)

print(le_attack.classes_)  # confirms order: should be [1 2 3 4]

stage2_models = {
    'DT': DecisionTreeClassifier(random_state=42),
    'RF': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGB': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss'),
    'LGBM': LGBMClassifier(random_state=42, n_jobs=-1)
}

stage2_results = {}
stage2_preds = {}
stage2_fitted = {}

for name, model in stage2_models.items():
    print(f"\n--- Stage 2: {name} ---")
    t0 = time.time()
    model.fit(X_train_attack, y_train_attack_enc)
    train_time = time.time() - t0
    stage2_fitted[name] = model

    y_pred_enc = model.predict(X_test_attack)
    y_pred = le_attack.inverse_transform(y_pred_enc)  # back to original 1-4 labels
    stage2_preds[name] = y_pred

    acc = accuracy_score(y_test_attack, y_pred)
    prec = precision_score(y_test_attack, y_pred, average='macro')
    rec = recall_score(y_test_attack, y_pred, average='macro')
    f1 = f1_score(y_test_attack, y_pred, average='macro')

    labels_present = sorted(y_test_attack.unique())
    cm = confusion_matrix(y_test_attack, y_pred, labels=labels_present)

    stage2_results[name] = {
        'accuracy': acc, 'precision_macro': prec, 'recall_macro': rec, 'f1_macro': f1,
        'confusion_matrix': cm.tolist(), 'labels': [LABEL_MAP[l] for l in labels_present],
        'train_time_sec': train_time
    }

    print(f"Accuracy: {acc:.4f} | Precision(macro): {prec:.4f} | Recall(macro): {rec:.4f} | F1(macro): {f1:.4f}")
    print(f"Confusion Matrix {[LABEL_MAP[l] for l in labels_present]}:\n{cm}")
    print(classification_report(y_test_attack, y_pred,
                                 target_names=[LABEL_MAP[l] for l in labels_present]))

[1 2 3 4]

--- Stage 2: DT ---
Accuracy: 0.8653 | Precision(macro): 0.8452 | Recall(macro): 0.8444 | F1(macro): 0.8448
Confusion Matrix ['DoS', 'DDoS', 'Mirai', 'Spoofing']:
[[52854  1253  4287  1546]
 [ 1305 57151   655   663]
 [ 4502   649 25807  3893]
 [ 1680   656  3962 25092]]
              precision    recall  f1-score   support

         DoS       0.88      0.88      0.88     59940
        DDoS       0.96      0.96      0.96     59774
       Mirai       0.74      0.74      0.74     34851
    Spoofing       0.80      0.80      0.80     31390

    accuracy                           0.87    185955
   macro avg       0.85      0.84      0.84    185955
weighted avg       0.87      0.87      0.87    185955


--- Stage 2: RF ---
Accuracy: 0.8835 | Precision(macro): 0.8649 | Recall(macro): 0.8705 | F1(macro): 0.8673
Confusion Matrix ['DoS', 'DDoS', 'Mirai', 'Spoofing']:
[[52544  1021  4862  1513]
 [ 1263 57190   681   640]
 [ 2487   415 28340  3609]
 [ 1130   534  3501 26225]]
         

In [9]:
# final complete hierarchial pipeline
model_names = ['DT', 'RF', 'XGB', 'LGBM']
hierarchical_results = {}

for name in model_names:
    print(f"\n=== Hierarchical {name} ===")

    stage1_pred = stage1_preds[name]          # 'Benign' / 'Attack' for every test row
    stage2_model = stage2_fitted[name]         # already trained on attack-only rows

    # Final prediction array, same length as full y_test
    final_pred = np.empty(len(y_test), dtype=object)

    # Wherever stage 1 says Benign -> final = Benign (label 0)
    benign_idx = np.where(stage1_pred == 'Benign')[0]
    final_pred[benign_idx] = 0

    # Wherever stage 1 says Attack -> run stage 2 model on those rows
    attack_idx = np.where(stage1_pred == 'Attack')[0]
    if len(attack_idx) > 0:
        stage2_input = X_test.iloc[attack_idx]
        stage2_final_pred_enc = stage2_model.predict(stage2_input)
        stage2_final_pred = le_attack.inverse_transform(stage2_final_pred_enc)
        final_pred[attack_idx] = stage2_final_pred

    final_pred = final_pred.astype(int)
    y_test_arr = y_test.values

    acc = accuracy_score(y_test_arr, final_pred)
    prec_macro = precision_score(y_test_arr, final_pred, average='macro')
    rec_macro = recall_score(y_test_arr, final_pred, average='macro')
    f1_macro = f1_score(y_test_arr, final_pred, average='macro')
    f1_weighted = f1_score(y_test_arr, final_pred, average='weighted')

    all_labels = sorted(LABEL_MAP.keys())
    cm = confusion_matrix(y_test_arr, final_pred, labels=all_labels)

    # FNR: attacks (label != 0) predicted as Benign (label 0)
    actual_attack_mask = y_test_arr != 0
    fn_count = np.sum((y_test_arr != 0) & (final_pred == 0))
    fnr = fn_count / actual_attack_mask.sum()

    hierarchical_results[name] = {
        'accuracy': acc,
        'precision_macro': prec_macro,
        'recall_macro': rec_macro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm.tolist(),
        'labels': [LABEL_MAP[l] for l in all_labels],
        'false_negative_rate': fnr
    }

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f}")
    print(f"False Negative Rate: {fnr:.4f}")
    print(f"5-class Confusion Matrix {[LABEL_MAP[l] for l in all_labels]}:\n{cm}")
    print(classification_report(y_test_arr, final_pred,
                                 target_names=[LABEL_MAP[l] for l in all_labels]))


=== Hierarchical DT ===
Accuracy: 0.8170
Macro F1: 0.7873 | Weighted F1: 0.8169
False Negative Rate: 0.0751
5-class Confusion Matrix ['Benign', 'DoS', 'DDoS', 'Mirai', 'Spoofing']:
[[65028  2669   398  4126  7353]
 [ 2506 51448  1128  3844  1014]
 [  381  1177 57053   580   583]
 [ 3883  4042   598 23709  2619]
 [ 7198  1183   578  2737 19694]]
              precision    recall  f1-score   support

      Benign       0.82      0.82      0.82     79574
         DoS       0.85      0.86      0.85     59940
        DDoS       0.95      0.95      0.95     59774
       Mirai       0.68      0.68      0.68     34851
    Spoofing       0.63      0.63      0.63     31390

    accuracy                           0.82    265529
   macro avg       0.79      0.79      0.79    265529
weighted avg       0.82      0.82      0.82    265529


=== Hierarchical RF ===
Accuracy: 0.8398
Macro F1: 0.8137 | Weighted F1: 0.8402
False Negative Rate: 0.0744
5-class Confusion Matrix ['Benign', 'DoS', 'DDoS', 'Mi

In [10]:
def make_json_safe(d):
    if isinstance(d, dict):
        return {k: make_json_safe(v) for k, v in d.items()}
    if isinstance(d, list):
        return [make_json_safe(v) for v in d]
    if isinstance(d, (np.integer,)):
        return int(d)
    if isinstance(d, (np.floating,)):
        return float(d)
    return d

all_results = {
    'stage1_binary': make_json_safe(stage1_results),
    'stage2_attack_type': make_json_safe(stage2_results),
    'hierarchical_combined': make_json_safe(hierarchical_results)
}

with open('../results/hierarchical_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Saved to ../results/hierarchical_results.json")

Saved to ../results/hierarchical_results.json
